# ManiBench: End-to-End Evaluation on Kaggle GPU (T4 / Dual T4×2)

### One command, publication-ready results

This notebook is a **thin launcher**. All the work — dependency bootstrap, NumPy-safe
install, model preflight, dual-GPU inference, the 8 evaluation metrics, LaTeX tables,
300-DPI figures and the Overleaf-ready `.zip` — now lives in a single resumable script:

```bash
python scripts/kaggle_run.py --preset paper
```

**What the runner handles for you**

| Phase | What happens |
| :--- | :--- |
| 1 | Environment detection (Kaggle / Colab / local, GPU inventory) |
| 2 | `apt` + `pip` bootstrap with **NumPy/PyTorch pinned via constraints** (fixes the `numpy._core.umath` `_center` bug) |
| 3 | Repo resolution (shallow clone if you only uploaded this notebook) |
| 4 | Dataset + model registry resolution |
| 5 | Hugging Face preflight — missing/gated repos are skipped with a reason instead of crashing |
| 6 | Benchmark execution with crash-safe checkpointing, FP16 sharding over both T4s, and a 5-step VRAM teardown between models |
| 7 | `fig1`/`fig2` publication figures (PDF + PNG, 300 DPI) |
| 8 | `manibench_paper_submission_bundle.zip` + `run_manifest.json` |

**Resumable by design.** Kaggle sessions are pre-empted constantly; every finished
trial is flushed to `results/kaggle_paper_run/checkpoint_records.jsonl`, so re-running
the same cell continues where it stopped instead of starting over.

**Presets** — `paper` (8 headline models × 12 tasks, rendered) · `full` (entire registry)
· `quick` (1 model, 2 tasks, no render) · `smoke` / `smoke-render` (synthetic code,
no weights — verifies the pipeline and the Cairo/FFmpeg renderer).

> **Before the first Kaggle run:** the runner must exist in the repository that the
> setup cell clones. Push this revision (or point `REPO_URL` in the setup cell at your
> own fork/branch). Running the notebook from a local clone works immediately.

## Step 1 — Configure the run

Edit the values below (defaults reproduce the paper run), or leave them as they are
and just execute the next two cells.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ManiBench Kaggle configuration - the only cell you normally need to edit
# ══════════════════════════════════════════════════════════════════════════
PRESET = "paper"           # paper | trio | full | quick | smoke | smoke-render
                           #   trio = base vs SFT-merged vs GRPO-adapter (Qwen3-8B)

MODELS = None              # None = preset default, or e.g. ["Qwen/Qwen3-8B"]
PROBLEMS = None            # None = all 12 problems, or e.g. ["MB-001", "MB-005"]
TRIALS = None              # None = preset default (1)
STRATEGY = None            # None = "zero_shot" (also: version_aware, cot)

SKIP_RENDER = None         # None = preset default; True = static metrics only
LOAD_IN_4BIT = False       # True for 31B models
COMPUTE_VISUAL_SIM = False # DINOv2 + DTW / SSIM against the reference clips

OUTPUT_DIR = "results/kaggle_paper_run"
EXTRA_ARGS = []            # e.g. ["--fresh", "--seed", "42", "--timeout", "90"]

print(f"preset={PRESET} models={MODELS or 'preset'} problems={PROBLEMS or 'all'} "
      f"trials={TRIALS or 'preset'} render={not SKIP_RENDER}")

## Step 2 — Run the whole benchmark

The next two cells clone the repository (if needed) and execute the complete pipeline.
A full 8-model × 12-task run takes roughly 3–4 hours on dual T4s; progress is
checkpointed continuously, and re-running the cell resumes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Locate (or clone) the ManiBench repository and install nothing yourself:
#  scripts/kaggle_run.py performs the NumPy-safe dependency bootstrap.
#  Point REPO_URL at your own fork/branch if you have not pushed upstream yet.
# ══════════════════════════════════════════════════════════════════════════
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nabin2004/ManiBench.git"
RUNNER_REL = Path("scripts/kaggle_run.py")
REPO_DIR = Path("/kaggle/working/ManiBench") if Path("/kaggle/working").exists() else Path.cwd() / "ManiBench"

if not RUNNER_REL.exists():
    if not (REPO_DIR / RUNNER_REL).exists():
        print(f"Cloning ManiBench into {REPO_DIR} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

RUNNER = RUNNER_REL.resolve()
if not RUNNER.exists():
    raise SystemExit(
        f"{RUNNER} not found.\n"
        f"The one-command runner lives in the latest revision of the repository.\n"
        f"Push the updated repo (or set REPO_URL to your fork/branch above) and re-run this cell."
    )

print(f"Repository root : {Path.cwd()}")
print(f"Runner          : {RUNNER}")
print(f"Python          : {sys.executable}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Execute the complete pipeline (install -> preflight -> benchmark -> paper)
# ══════════════════════════════════════════════════════════════════════════
cmd = [sys.executable, str(RUNNER), "--preset", PRESET, "--output-dir", OUTPUT_DIR]
if MODELS:
    cmd += ["--models", *MODELS]
if PROBLEMS:
    cmd += ["--problems", *PROBLEMS]
if TRIALS:
    cmd += ["--trials", str(TRIALS)]
if STRATEGY:
    cmd += ["--strategy", STRATEGY]
if SKIP_RENDER:
    cmd += ["--skip-render"]
if LOAD_IN_4BIT:
    cmd += ["--load-in-4bit"]
if COMPUTE_VISUAL_SIM:
    cmd += ["--compute-visual-sim", "--with-vision"]
cmd += EXTRA_ARGS

print("$ " + " ".join(cmd) + "\n", flush=True)
exit_code = subprocess.run(cmd).returncode
print(f"\nManiBench runner finished with exit code {exit_code}")
if exit_code != 0:
    print("Tip: the checkpoint file lets you simply re-run this cell to resume.")

## Step 3 — Inspect the paper artifacts

The leaderboard, styled summary table, LaTeX tables and figures are all generated by
the runner. This cell simply displays them.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Display the generated artifacts
# ══════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

out = Path(OUTPUT_DIR)
tables = out / "tables"

manifest_path = out / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    counts = manifest.get("counts", {})
    print(f"Evaluations recorded : {counts.get('records', 0)}")
    print(f"Wall clock           : {manifest.get('elapsed_seconds', 0) / 60:.1f} min")
    skipped = manifest.get("config", {}).get("skipped_models", [])
    if skipped:
        print("\nSkipped models:")
        for entry in skipped:
            print(f"  - {entry['short_name']}: {entry['reason'][:90]}")

leaderboard = tables / "leaderboard.md"
if leaderboard.exists():
    display(Markdown(leaderboard.read_text(encoding="utf-8")))

summary_csv = tables / "results_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    cols = [c for c in ["short_name", "family", "training_method", "pass_rate_pct",
                        "vcer_pct", "mas_mean", "cmi_mean", "car_mean",
                        "alignment_mean", "coverage_mean"] if c in df.columns]
    styled = df[cols].style.background_gradient(subset=["pass_rate_pct"], cmap="Blues")
    if "vcer_pct" in cols:
        styled = styled.background_gradient(subset=["vcer_pct"], cmap="Reds_r")
    display(styled)

for fig in sorted((out / "figures").glob("*.png")) if (out / "figures").exists() else []:
    print(fig.name)
    display(Image(filename=str(fig)))

table1 = tables / "table1_main_benchmark.tex"
if table1.exists():
    print("\n" + "=" * 75)
    print("  LaTeX Table 1 (main results) - ready for Overleaf")
    print("=" * 75)
    print(table1.read_text(encoding="utf-8"))

## Step 4 — Download the submission bundle

Everything needed for the paper is inside `manibench_paper_submission_bundle.zip`
(LaTeX `booktabs` tables, per-trial CSV, the raw JSON archive, the figures and the
draft results prose).

Open Kaggle's **Output** panel (right-hand file browser) and download the zip, then drop
its contents straight into your Overleaf project.

### Useful flags for later runs

| Goal | Flag |
| :--- | :--- |
| Start a completely fresh run | `--fresh` |
| Resume after a pre-emption | *(default — just re-run)* |
| Only 2 problems, quick check | `--problems MB-001 MB-005` |
| 3 stochastic trials per task | `--trials 3` |
| Skip rendering (static metrics only) | `--skip-render` |
| 4-bit quantization (31B models) | `--load-in-4bit` |
| DINOv2 + DTW visual similarity | `--compute-visual-sim --with-vision` |
| Any other Hugging Face model | `--models your-org/your-model` |

Run `python scripts/kaggle_run.py --help` inside the notebook for the full list.